In [1]:
import sys
sys.path.append('../src')
import pandas as pd
from detector import detect_missing

df = pd.read_csv('../data/raw/air_quality.csv')
print("Dataset loaded!", df.shape)

Dataset loaded! (29531, 16)


In [2]:
report = detect_missing(df)

for col, info in report.items():
    print(f"{col:20} | {info['missing_pct']:6.2f}% | {info['severity']}")

KeyError: 'missing_pct'

In [3]:
# Check what keys your report actually has
first_col = list(report.keys())[0]
print(report[first_col])

{'count': 0, 'pct': 0.0, 'severity': 'OK'}


In [4]:
for col, info in report.items():
    print(f"{col:20} | {info['pct']:6.2f}% | {info['severity']}")

City                 |   0.00% | OK
Date                 |   0.00% | OK
PM2.5                |  15.57% | MEDIUM
PM10                 |  37.72% | CRITICAL
NO                   |  12.13% | MEDIUM
NO2                  |  12.14% | MEDIUM
NOx                  |  14.17% | MEDIUM
NH3                  |  34.97% | CRITICAL
CO                   |   6.97% | LOW
SO2                  |  13.05% | MEDIUM
O3                   |  13.62% | MEDIUM
Benzene              |  19.04% | MEDIUM
Toluene              |  27.23% | HIGH
Xylene               |  61.32% | CRITICAL
AQI                  |  15.85% | MEDIUM
AQI_Bucket           |  15.85% | MEDIUM


In [6]:
print("🚨 CRITICAL FLAWS FOUND:\n")
for col, info in report.items():
    if info['severity'] == "CRITICAL":
        print(f"  → {col}: {info['pct']}% missing ({info['count']} rows)")

🚨 CRITICAL FLAWS FOUND:

  → PM10: 37.72% missing (11140 rows)
  → NH3: 34.97% missing (10328 rows)
  → Xylene: 61.32% missing (18109 rows)


In [4]:
import numpy as np

def detect_outliers(df, col):
    data = df[col].dropna()  # ignore missing values for now
    
    # ── IQR Method ──────────────────────────
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    iqr_outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    
    # ── Z-Score Method ───────────────────────
    z_scores = (data - data.mean()) / data.std()
    zscore_outliers = z_scores[abs(z_scores) > 3]
    
    return {
        "col": col,
        "iqr_outliers": len(iqr_outliers),
        "zscore_outliers": len(zscore_outliers),
        "iqr_bounds": (round(Q1 - 1.5*IQR, 2), round(Q3 + 1.5*IQR, 2))
    }

In [6]:
# Test it on PM2.5
result = detect_outliers(df, 'PM2.5')
print(result)

{'col': 'PM2.5', 'iqr_outliers': 1982, 'zscore_outliers': 497, 'iqr_bounds': (np.float64(-48.84), np.float64(158.24))}


In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/air_quality.csv')
print("Loaded!", df.shape)

Loaded! (29531, 16)


In [7]:
pollutant_cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']

print(f"{'COLUMN':<15} {'IQR OUTLIERS':>15} {'ZSCORE OUTLIERS':>17}")
print("=" * 50)

for col in pollutant_cols:
    result = detect_outliers(df, col)
    print(f"{col:<15} {result['iqr_outliers']:>15} {result['zscore_outliers']:>17}")

COLUMN             IQR OUTLIERS   ZSCORE OUTLIERS
PM2.5                      1982               497
PM10                       1057               358
NO                         2459               595
NO2                        1188               469
NOx                        1868               562
NH3                        1015               355
CO                         2475               546
SO2                        2578               581
O3                          713               346
Benzene                    1668                97
Toluene                    2427               242
Xylene                     1119               186


In [8]:
outlier_report = {}

for col in pollutant_cols:
    result = detect_outliers(df, col)
    outlier_report[col] = {
        "iqr_outliers": result['iqr_outliers'],
        "zscore_outliers": result['zscore_outliers'],
        "bounds": result['iqr_bounds']
    }

print("✅ Outlier report built for", len(outlier_report), "columns")

✅ Outlier report built for 12 columns
